# Kaiwa Transcription (GPU + large-v3-turbo)
# 
# Model: large-v3-turbo (best quality/speed balance for Japanese)
# - large-v3 equivalent accuracy, 2x faster
# - float16 on T4 GPU (16GB VRAM, model uses ~6GB)
# - VAD filter enabled (skip silence, reduce hallucination)
# - beam_size=5 for accuracy
#
# Steps:
# 1. Runtime > Change runtime type > T4 GPU
# 2. Run all cells in order
# 3. Upload .m4a files when prompted
# 4. Wait for transcription
# 5. Download zip

In [ ]:
!pip install -q faster-whisper
!mkdir -p /content/kaiwa /content/transcripts

In [ ]:
# Upload files - run this cell, then click the upload button
from google.colab import files
import shutil, os

uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, f'/content/kaiwa/{fname}')
print(f'Uploaded {len(uploaded)} files')

In [ ]:
# OR: Mount Google Drive (if files are in Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# INPUT_DIR = '/content/drive/MyDrive/kaiwa'  # adjust path

In [ ]:
import os, time
from faster_whisper import WhisperModel

INPUT_DIR = '/content/kaiwa'
OUTPUT_DIR = '/content/transcripts'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# large-v3-turbo: best Japanese accuracy at reasonable speed
# float16 on T4 GPU, ~6GB VRAM
model = WhisperModel('large-v3-turbo', device='cuda', compute_type='float16')
print('Model loaded: large-v3-turbo (GPU, float16)')

files_list = sorted([f for f in os.listdir(INPUT_DIR) if f.endswith('.m4a')])
print(f'Found {len(files_list)} files\n')

total_time = 0
for i, fname in enumerate(files_list):
    fpath = os.path.join(INPUT_DIR, fname)
    out_name = fname.replace('.m4a', '.txt')
    out_path = os.path.join(OUTPUT_DIR, out_name)

    if os.path.exists(out_path):
        print(f'[{i+1}/{len(files_list)}] SKIP (already done): {fname}')
        continue

    print(f'[{i+1}/{len(files_list)}] {fname} ...', end=' ', flush=True)
    t0 = time.time()

    segments, info = model.transcribe(
        fpath,
        language='ja',
        beam_size=5,
        vad_filter=True,              # skip silence, reduce hallucination
        vad_parameters=dict(
            min_silence_duration_ms=500,  # 0.5s silence = segment break
        ),
        condition_on_previous_text=True,  # use context for better accuracy
    )

    lines = []
    for seg in segments:
        text = seg.text.strip()
        if text:
            lines.append(text)

    transcript = '\n'.join(lines)
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(transcript)

    elapsed = time.time() - t0
    total_time += elapsed
    print(f'{elapsed:.1f}s ({len(lines)} segments)')

print(f'\nAll done! Total: {total_time:.0f}s ({total_time/60:.1f}min)')

In [ ]:
# Download all transcripts as zip
import zipfile
from google.colab import files

with zipfile.ZipFile('/content/transcripts.zip', 'w') as zf:
    for fname in os.listdir(OUTPUT_DIR):
        zf.write(os.path.join(OUTPUT_DIR, fname), fname)

files.download('/content/transcripts.zip')
print('Download started!')

In [ ]:
# Preview transcripts
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    with open(fpath, 'r', encoding='utf-8') as f:
        content = f.read()
    print(f'\n{"="*60}')
    print(f'FILE: {fname}')
    print(f'{"="*60}')
    print(content[:500])
    if len(content) > 500:
        print(f'... ({len(content)} chars total)')